# OpenAI Agents SDK: From Fundamentals to Production-Ready AI Agents

## Notebook 1.3 — Anatomy of a Modern AI Agent
### Part 2: Reasoning, Planning, Reflection, and Failure Recovery

---

**Prerequisite:** Notebook 1.3 — Part 1

> **Central question**
>
> How does an AI agent decide what to do next, create a plan, adapt when the environment changes, evaluate progress, and recover from failure?

Part 1 introduced the main components of an agent:

```text
Goal → Instructions → Model → Runtime → Tools → Observations → State
```

Part 2 focuses on the decision-making behaviour inside that architecture.

We will study:

- reasoning as observable task behaviour;
- planning and task decomposition;
- task dependencies;
- dynamic re-planning;
- reflection and critique;
- retry strategies;
- evaluator patterns;
- failure recovery;
- and plan–act–evaluate loops.

The examples deliberately begin with deterministic Python logic before introducing LLM-oriented patterns.

## Learning Outcomes

By the end of this notebook, you should be able to:

1. distinguish reasoning, planning, acting, and evaluating;
2. explain why complex goals require decomposition;
3. represent plans using lists, states, and dependency graphs;
4. distinguish static plans from adaptive plans;
5. identify conditions that should trigger re-planning;
6. explain reflection without relying on hidden internal reasoning;
7. design observable self-checking behaviour;
8. implement retry strategies with limits;
9. design evaluator functions and quality gates;
10. build a complete plan–act–evaluate loop in Python;
11. analyse common failure modes in agentic systems;
12. identify when human intervention is safer than further autonomy.

>
> Agent design should focus on observable decisions and verifiable outputs. Avoid framing reliability as “the model thought harder”. Reliability comes from architecture, constraints, state, evaluation, and feedback.

## 1. Recap: The Agent Loop

Part 1 introduced:

```text
Observe → Decide → Act → Observe Again
```

For complex tasks, the “decide” step often contains several sub-processes:

```text
Observe
   ↓
Interpret current state
   ↓
Reason about options
   ↓
Create or update plan
   ↓
Select next action
   ↓
Act
   ↓
Evaluate result
   ↓
Continue, revise, or stop
```

A more detailed loop is therefore:

```text
OBSERVE → REASON → PLAN → ACT → EVALUATE → RE-PLAN OR STOP
```

These stages may be implemented by:

- a language model;
- deterministic software;
- specialised evaluators;
- human approval;
- or a combination.

## 2. What Do We Mean by “Reasoning”?

In practical agent engineering, reasoning means:

> Using available information to select a useful next step.

We should not rely on hidden internal thought as an engineering control.

Instead, we inspect observable artefacts such as:

- selected action;
- action arguments;
- plan;
- assumptions;
- evidence used;
- confidence or uncertainty;
- tool results;
- validation results;
- and stopping decision.

Example:

```text
Observation:
The user wants a meeting next week.
Monday is unavailable.
Tuesday has two free slots.

Observable decision:
Check the attendees' calendars for Tuesday.
```

The useful engineering object is the decision, not private internal reasoning.

## 3. Reasoning vs Planning

These terms are related but not identical.

### Reasoning

Evaluates the current situation and chooses among possibilities.

Example:

```text
The room holds only 60 people.
The event expects 100 people.
Therefore, this room is unsuitable.
```

### Planning

Organises a sequence of actions required to reach a goal.

Example:

```text
1. Check dates
2. Check room capacity
3. Confirm speaker availability
4. Estimate budget
5. Request approval
```

### Relationship

```text
Reasoning helps create and revise plans.
Planning structures future actions.
```

## 4. Acting vs Evaluating

### Acting

The agent performs an operation:

- search;
- calculate;
- send;
- update;
- read;
- write;
- call an API.

### Evaluating

The system checks whether the result is useful, valid, safe, or complete.

Example:

```text
Action:
Generate an event agenda.

Evaluation:
Does the agenda fit the available 120 minutes?
Does it include a break?
Does it cover all required topics?
```

Without evaluation, an agent may perform actions without confirming progress.

## Knowledge Check 1

1. What is reasoning in practical agent engineering?
2. Why should observable decisions matter more than hidden internal thought?
3. What is the difference between reasoning and planning?
4. Give one example of acting and one example of evaluating.
5. Why might an agent require evaluation after every important tool call?
6. What could happen if a system plans but never re-evaluates?

## 5. Why Complex Goals Need Decomposition

Consider:

> “Launch a new online course.”

This goal hides many tasks:

```text
Define audience
Design curriculum
Create lessons
Build landing page
Configure payment
Prepare marketing
Test enrolment flow
Schedule launch
Monitor registrations
```

Trying to solve the entire goal in one step creates several problems:

- important requirements may be forgotten;
- dependencies may be ignored;
- failures become difficult to locate;
- progress cannot be measured;
- retries may repeat too much work.

Decomposition converts a large problem into manageable units.

## 6. Good and Poor Decomposition

### Poor decomposition

```text
1. Plan course
2. Build course
3. Launch course
```

These steps are still too broad.

### Better decomposition

```text
1. Define target learners
2. Define learning outcomes
3. Create module structure
4. Draft lesson content
5. Record or prepare delivery material
6. Build enrolment page
7. Test payment and registration
8. Prepare launch communication
9. Request approval
10. Publish
```

A useful task should be:

- clear;
- actionable;
- measurable;
- bounded;
- and connected to the goal.

## 7. Atomic Tasks

An **atomic task** is small enough to execute and verify independently.

Example:

```text
Broad:
Prepare marketing.

Atomic:
Draft a 150-word launch email.
```

Atomic tasks help with:

- tool selection;
- validation;
- retries;
- progress tracking;
- delegation;
- and error isolation.

But excessive decomposition creates overhead.

```text
Too broad → hard to execute
Too narrow → too many steps
```

Good decomposition balances clarity and efficiency.

In [ ]:
goal = "Launch an online AI agents course"

poor_plan = [
    "Plan course",
    "Build course",
    "Launch course"
]

better_plan = [
    "Define target learners",
    "Write learning outcomes",
    "Create module structure",
    "Draft lesson content",
    "Build enrolment page",
    "Test registration flow",
    "Prepare launch email",
    "Request approval",
    "Publish the course"
]

print("POOR PLAN")
for task in poor_plan:
    print(" -", task)

print("\nBETTER PLAN")
for index, task in enumerate(better_plan, start=1):
    print(f"{index}. {task}")

## 8. Representing a Plan

A plan can be stored as structured data.

```python
{
    "goal": "Launch course",
    "tasks": [
        {
            "id": "T1",
            "name": "Define target learners",
            "status": "pending"
        }
    ]
}
```

Structured plans support:

- task status;
- dependencies;
- assigned tools;
- priority;
- retry count;
- evidence;
- and completion checks.

In [ ]:
plan = {
    "goal": "Launch an online AI agents course",
    "tasks": [
        {
            "id": "T1",
            "name": "Define target learners",
            "status": "pending",
            "priority": "high",
            "depends_on": []
        },
        {
            "id": "T2",
            "name": "Write learning outcomes",
            "status": "pending",
            "priority": "high",
            "depends_on": ["T1"]
        },
        {
            "id": "T3",
            "name": "Create module structure",
            "status": "pending",
            "priority": "high",
            "depends_on": ["T2"]
        }
    ]
}

plan

## 9. Task Dependencies

Some tasks cannot begin until others finish.

Example:

```text
Define audience
      ↓
Write learning outcomes
      ↓
Create curriculum
      ↓
Draft lessons
```

Another branch may run in parallel:

```text
Choose platform
      ↓
Configure enrolment
```

Dependencies create a task graph rather than a simple list.

```text
             ┌───────────────┐
             │Define Audience│
             └───────┬───────┘
                     ▼
             ┌───────────────┐
             │Learning Outcomes│
             └───────┬───────┘
                     ▼
              ┌──────────────┐
              │ Curriculum   │
              └──────────────┘

Choose Platform → Configure Enrolment
```

## 10. Ready Tasks

A task is ready when:

1. its status is pending;
2. all dependencies are complete;
3. required information is available;
4. permissions allow execution.

```text
Ready(task) =
Pending
AND dependencies complete
AND required inputs present
AND authorised
```

This is more reliable than always taking the first item in a list.

In [ ]:
def get_task(plan: dict, task_id: str) -> dict:
    for task in plan["tasks"]:
        if task["id"] == task_id:
            return task
    raise KeyError(task_id)


def dependencies_complete(plan: dict, task: dict) -> bool:
    for dependency_id in task["depends_on"]:
        dependency = get_task(plan, dependency_id)
        if dependency["status"] != "completed":
            return False
    return True


def ready_tasks(plan: dict) -> list[dict]:
    return [
        task
        for task in plan["tasks"]
        if task["status"] == "pending"
        and dependencies_complete(plan, task)
    ]


ready_tasks(plan)

## 11. Interpreting Ready-Task Selection

Only `T1` is initially ready because:

- `T2` depends on `T1`;
- `T3` depends on `T2`.

After `T1` completes, `T2` becomes ready.

This mechanism prevents the agent from executing tasks in an invalid order.

In [ ]:
get_task(plan, "T1")["status"] = "completed"
ready_tasks(plan)

## 12. Sequential and Parallel Plans

### Sequential plan

```text
A → B → C → D
```

Each task depends on the previous one.

### Parallel plan

```text
        ┌→ B ─┐
A ──────┤     ├→ D
        └→ C ─┘
```

Tasks B and C can run independently.

Parallel execution can reduce latency, but it introduces:

- concurrency;
- conflicting updates;
- dependency management;
- rate limits;
- shared-state risks.

Early agent systems should often prefer simple sequential execution unless parallelism provides clear value.

## Knowledge Check 2

1. Why decompose complex goals?
2. What is an atomic task?
3. What makes a decomposition too broad?
4. What makes a decomposition too narrow?
5. Why are dependencies important?
6. What conditions make a task ready?
7. When might parallel execution be useful?
8. What risks appear with parallel execution?

## 13. Static Planning

A static plan is created once and executed without modification.

```text
Create Plan
    ↓
Execute Task 1
    ↓
Execute Task 2
    ↓
Execute Task 3
    ↓
Finish
```

Static plans work well when:

- the environment is predictable;
- requirements are complete;
- tools are reliable;
- the process is stable.

Examples:

- data export;
- report formatting;
- scheduled backups;
- standard onboarding workflow.

## 14. Dynamic Planning

A dynamic plan changes in response to new observations.

Example:

```text
Original plan:
Book Room A

Observation:
Room A unavailable

Revised plan:
1. Search alternative rooms
2. Compare capacity
3. Update budget
4. Request approval if cost increases
```

Dynamic planning is valuable when:

- information is incomplete;
- tools may fail;
- environments change;
- goals are open-ended;
- multiple valid paths exist.

## 15. Re-Planning Triggers

A system should not revise plans randomly.

Useful re-planning triggers include:

- required resource unavailable;
- tool call fails;
- user changes requirements;
- new constraint appears;
- evaluator rejects output;
- budget or time limit exceeded;
- dependency becomes invalid;
- task is no longer necessary;
- human approval modifies scope.

```text
Observation
   ↓
Does it invalidate the current plan?
   ├── No → continue
   └── Yes → revise plan
```

In [ ]:
def should_replan(observation: dict) -> bool:
    replanning_events = {
        "resource_unavailable",
        "requirement_changed",
        "tool_failure",
        "evaluation_failed",
        "budget_exceeded"
    }
    return observation.get("type") in replanning_events


observations = [
    {"type": "task_completed", "message": "Agenda drafted."},
    {"type": "resource_unavailable", "message": "Main auditorium unavailable."},
    {"type": "requirement_changed", "message": "Expected attendance increased."}
]

for obs in observations:
    print(obs["type"], "→ re-plan?", should_replan(obs))

## 16. Local Re-Planning vs Global Re-Planning

### Local re-planning

Change only the affected task.

Example:

```text
Room unavailable
→ replace room-search task
```

### Global re-planning

Reconsider the entire plan.

Example:

```text
Attendance doubles
→ room, budget, agenda, catering, and staffing all change
```

Local re-planning is usually cheaper and safer.

Global re-planning is necessary when a new observation affects the entire strategy.

## 17. Preserving Completed Work

A poor agent may rebuild everything after one failure.

A better agent preserves valid completed work.

```text
Completed:
✓ Learning outcomes
✓ Module structure

Failed:
✗ Video hosting setup

Re-plan:
Replace only hosting-related tasks
```

This principle is called **minimal repair**.

```text
Repair the smallest valid portion of the plan.
```

Minimal repair reduces:

- repeated cost;
- duplicate actions;
- user frustration;
- inconsistency;
- and unnecessary risk.

In [ ]:
project_state = {
    "tasks": [
        {"id": "T1", "name": "Define outcomes", "status": "completed"},
        {"id": "T2", "name": "Create modules", "status": "completed"},
        {"id": "T3", "name": "Configure video hosting", "status": "failed"},
        {"id": "T4", "name": "Publish course", "status": "blocked"}
    ]
}


def minimal_repair(state: dict) -> list[dict]:
    repaired_tasks = []

    for task in state["tasks"]:
        if task["status"] == "failed":
            repaired_tasks.append({
                "id": f"{task['id']}-R1",
                "name": f"Find alternative for: {task['name']}",
                "status": "pending"
            })

    return repaired_tasks


minimal_repair(project_state)

## 18. Plan Versioning

When a plan changes, store versions.

```text
Plan v1:
Use Auditorium A

Plan v2:
Use Seminar Hall B

Reason:
Auditorium A unavailable
```

Versioning helps with:

- traceability;
- debugging;
- user explanation;
- evaluation;
- rollback.

A plan change should record:

- what changed;
- why it changed;
- which observation triggered it;
- who approved it;
- and which tasks were preserved.

In [ ]:
plan_versions = [
    {
        "version": 1,
        "change": "Initial plan",
        "reason": "Created from user goal",
        "tasks": ["Check Auditorium A", "Draft agenda"]
    },
    {
        "version": 2,
        "change": "Replaced venue task",
        "reason": "Auditorium A unavailable",
        "tasks": ["Check Seminar Hall B", "Draft agenda"]
    }
]

for version in plan_versions:
    print(f"Plan v{version['version']}")
    print("Change:", version["change"])
    print("Reason:", version["reason"])
    print("Tasks:", version["tasks"])
    print()

## Knowledge Check 3

1. What is a static plan?
2. What is a dynamic plan?
3. Give four re-planning triggers.
4. What is local re-planning?
5. What is global re-planning?
6. Explain minimal repair.
7. Why preserve completed work?
8. Why should plan changes be versioned?

## 19. Reflection in Agent Systems

Reflection is often described vaguely as:

> “The agent thinks about its own answer.”

For engineering, a better definition is:

> The system evaluates an intermediate or final result against explicit criteria and uses the evaluation to decide whether to accept, revise, retry, or escalate.

Reflection can be implemented using:

- deterministic checks;
- another model call;
- a specialised evaluator model;
- tests;
- user feedback;
- human review.

## 20. Observable Reflection

An observable reflection record might contain:

```python
{
    "candidate_output": "...",
    "criteria": [
        "contains three recommendations",
        "uses official sources",
        "includes uncertainty"
    ],
    "evaluation": {
        "passed": False,
        "issues": ["Only two recommendations"]
    },
    "next_action": "revise"
}
```

This is useful because the system exposes:

- criteria;
- result;
- identified issue;
- chosen next action.

It does not require access to private hidden reasoning.

## 21. Deterministic Evaluation

Whenever possible, use deterministic checks.

Examples:

- valid JSON;
- required keys present;
- number within range;
- no missing fields;
- tests pass;
- file exists;
- total matches line items;
- output length below limit.

Deterministic evaluators are:

- consistent;
- fast;
- testable;
- explainable;
- and inexpensive.

In [ ]:
def evaluate_recommendations(result: dict) -> dict:
    issues = []

    recommendations = result.get("recommendations", [])

    if len(recommendations) != 3:
        issues.append("Exactly three recommendations are required.")

    if not result.get("sources"):
        issues.append("At least one source is required.")

    if "uncertainty" not in result:
        issues.append("Uncertainty statement is missing.")

    return {
        "passed": len(issues) == 0,
        "issues": issues
    }


candidate = {
    "recommendations": ["Option A", "Option B"],
    "sources": [],
}

evaluate_recommendations(candidate)

## 22. Model-Based Evaluation

Some qualities are difficult to check with simple rules.

Examples:

- clarity;
- relevance;
- completeness;
- tone;
- coherence;
- evidence quality;
- policy alignment.

A model-based evaluator may receive:

```text
Goal
Candidate output
Evaluation rubric
```

and return structured feedback.

However, model-based evaluation is probabilistic. It should be constrained by:

- a clear rubric;
- structured output;
- limited scope;
- deterministic checks where possible;
- human review for high-stakes decisions.

## 23. Evaluation Rubrics

A rubric converts vague quality into explicit criteria.

Example:

| Criterion | Score 0 | Score 1 | Score 2 |
|---|---|---|---|
| Completeness | Major sections missing | Some sections present | All required sections |
| Evidence | No sources | Weak sources | Strong relevant sources |
| Clarity | Difficult to follow | Mostly clear | Clear and concise |
| Safety | Violates policy | Uncertain | Fully compliant |

A result may be accepted only when:

```text
Total score ≥ 7
AND safety score = 2
```

Rubrics improve consistency.

In [ ]:
rubric_scores = {
    "completeness": 2,
    "evidence": 1,
    "clarity": 2,
    "safety": 2
}

total_score = sum(rubric_scores.values())
accepted = (
    total_score >= 7
    and rubric_scores["safety"] == 2
)

print("Scores:", rubric_scores)
print("Total:", total_score)
print("Accepted:", accepted)

## 24. Self-Evaluation vs Independent Evaluation

### Self-evaluation

The same model creates and evaluates the result.

Advantages:

- simple;
- low integration effort;
- shared context.

Risks:

- repeated blind spots;
- overly favourable evaluation;
- correlated failure.

### Independent evaluation

A separate evaluator checks the result.

Advantages:

- stronger separation;
- independent rubric;
- easier auditing.

Costs:

- more latency;
- additional model use;
- more orchestration.

High-stakes systems benefit from independent checks.

## 25. Reflection Outcomes

Evaluation should lead to a clear next action.

```text
PASS
→ accept output

MINOR ISSUE
→ revise locally

MAJOR ISSUE
→ re-plan

UNSAFE
→ stop or escalate

MISSING INFORMATION
→ ask user

TOOL FAILURE
→ retry or use alternative
```

Reflection is useful only when it influences control flow.

In [ ]:
def choose_evaluation_action(evaluation: dict) -> str:
    if evaluation.get("unsafe"):
        return "escalate"

    if evaluation.get("missing_information"):
        return "ask_user"

    severity = evaluation.get("severity", "none")

    if severity == "none":
        return "accept"
    if severity == "minor":
        return "revise"
    if severity == "major":
        return "replan"

    return "stop"


evaluations = [
    {"severity": "none"},
    {"severity": "minor"},
    {"severity": "major"},
    {"unsafe": True},
    {"missing_information": True}
]

for item in evaluations:
    print(item, "→", choose_evaluation_action(item))

## Knowledge Check 4

1. Define reflection in engineering terms.
2. Why are explicit criteria important?
3. Give five deterministic evaluation checks.
4. When is model-based evaluation useful?
5. What is a rubric?
6. Why might independent evaluation outperform self-evaluation?
7. What control-flow outcomes can follow evaluation?
8. Why is reflection useless if it does not change behaviour?

## 26. Retry Strategies

A retry repeats an operation after failure.

But not every failure should be retried.

### Retryable failures

- temporary timeout;
- rate limit;
- transient network error;
- service unavailable.

### Non-retryable failures

- invalid credentials;
- permission denied;
- malformed input;
- prohibited action;
- permanently missing resource.

Blind retries can increase cost and delay without improving success.

## 27. Retry Limits

A safe retry policy includes:

- maximum attempts;
- delay;
- backoff strategy;
- retryable error types;
- escalation condition.

Example:

```text
Attempt 1 → fail
Wait 1 second

Attempt 2 → fail
Wait 2 seconds

Attempt 3 → fail
Stop and escalate
```

This is exponential backoff.

In [ ]:
def retry_delays(max_attempts: int, base_delay: int = 1) -> list[int]:
    return [
        base_delay * (2 ** attempt)
        for attempt in range(max_attempts)
    ]


retry_delays(max_attempts=5)

## 28. Retry With Bounded Execution

The following example simulates a flaky tool.

It fails on the first two calls and succeeds on the third.

In [ ]:
class TemporaryToolError(Exception):
    pass


attempt_counter = {"count": 0}


def flaky_tool() -> str:
    attempt_counter["count"] += 1

    if attempt_counter["count"] < 3:
        raise TemporaryToolError(
            f"Temporary failure on attempt {attempt_counter['count']}"
        )

    return "Tool completed successfully."


def run_with_retries(tool, max_attempts: int = 3) -> dict:
    errors = []

    for attempt in range(1, max_attempts + 1):
        try:
            result = tool()
            return {
                "status": "success",
                "attempt": attempt,
                "result": result,
                "errors": errors
            }
        except TemporaryToolError as error:
            errors.append(str(error))

    return {
        "status": "failed",
        "attempt": max_attempts,
        "result": None,
        "errors": errors
    }


run_with_retries(flaky_tool)

## 29. Alternative Actions

Retrying the same tool is not always the best recovery strategy.

Example:

```text
Primary search tool fails
        ↓
Retry once
        ↓
Still fails
        ↓
Use backup search provider
        ↓
Still fails
        ↓
Ask user or stop
```

Recovery options include:

- retry;
- modify arguments;
- use alternate tool;
- skip optional task;
- ask user;
- request approval;
- escalate to human;
- stop safely.

## 30. Error Budgets

An agent should track cumulative failures.

```python
{
    "tool_errors": 2,
    "evaluation_failures": 1,
    "replans": 1
}
```

An error budget defines how much failure is tolerated before stopping.

Example:

```text
Maximum tool errors: 3
Maximum re-plans: 2
Maximum evaluation revisions: 2
```

This prevents infinite recovery loops.

In [ ]:
execution_limits = {
    "max_tool_errors": 3,
    "max_replans": 2,
    "max_revisions": 2
}

execution_counters = {
    "tool_errors": 2,
    "replans": 1,
    "revisions": 2
}


def limits_exceeded(limits: dict, counters: dict) -> list[str]:
    exceeded = []

    mapping = {
        "tool_errors": "max_tool_errors",
        "replans": "max_replans",
        "revisions": "max_revisions"
    }

    for counter_name, limit_name in mapping.items():
        if counters[counter_name] >= limits[limit_name]:
            exceeded.append(counter_name)

    return exceeded


limits_exceeded(execution_limits, execution_counters)

## 31. Failure Classification

Before recovery, classify the failure.

| Failure Type | Example | Likely Response |
|---|---|---|
| Transient | Timeout | Retry |
| Input error | Invalid date | Correct arguments |
| Missing data | No attendee count | Ask user |
| Resource conflict | Venue unavailable | Re-plan |
| Permission | No send access | Request approval or stop |
| Quality failure | Incomplete report | Revise |
| Safety failure | Prohibited action | Stop and escalate |
| Goal impossible | Deadline already passed | Explain limitation |

Classification improves recovery decisions.

## 32. Graceful Degradation

Sometimes the full goal cannot be completed.

A useful agent may still provide partial value.

Example:

```text
Unable to send invitations
because email access is unavailable.

Completed:
✓ drafted invitation
✓ identified recipients
✓ prepared subject line

Pending:
✗ sending email
```

Graceful degradation means:

- preserve completed work;
- state what failed;
- avoid false claims;
- provide next steps;
- return control to the user.

## Knowledge Check 5

1. Which failures should usually be retried?
2. Which failures should not be retried?
3. What is exponential backoff?
4. Why set retry limits?
5. What alternatives exist besides retrying?
6. What is an error budget?
7. Why classify failures?
8. What is graceful degradation?
9. Why must agents avoid claiming that failed actions succeeded?

## 33. The Plan–Act–Evaluate Loop

We can now combine the major concepts.

```text
GOAL
  ↓
CREATE PLAN
  ↓
SELECT READY TASK
  ↓
ACT
  ↓
OBSERVE RESULT
  ↓
EVALUATE
  ↓
┌───────────────────────────────┐
│ pass → mark complete          │
│ minor issue → revise          │
│ major issue → re-plan         │
│ transient error → retry       │
│ missing info → ask user       │
│ unsafe → stop/escalate        │
└───────────────────────────────┘
  ↓
REPEAT OR FINISH
```

## 34. Integrated Agent State

We will create a state object containing:

- goal;
- plan;
- current task;
- history;
- counters;
- limits;
- status;
- final output.

In [ ]:
agent_state = {
    "goal": "Prepare a validated workshop agenda",
    "tasks": [
        {
            "id": "T1",
            "name": "Collect workshop requirements",
            "status": "pending",
            "depends_on": []
        },
        {
            "id": "T2",
            "name": "Draft workshop agenda",
            "status": "pending",
            "depends_on": ["T1"]
        },
        {
            "id": "T3",
            "name": "Validate agenda",
            "status": "pending",
            "depends_on": ["T2"]
        }
    ],
    "requirements": {
        "duration_minutes": 120,
        "required_sections": [
            "Introduction",
            "Concept explanation",
            "Demonstration",
            "Activity",
            "Recap"
        ]
    },
    "agenda": None,
    "history": [],
    "counters": {
        "tool_errors": 0,
        "revisions": 0,
        "replans": 0
    },
    "limits": {
        "max_turns": 10,
        "max_tool_errors": 2,
        "max_revisions": 2,
        "max_replans": 1
    },
    "status": "running"
}

agent_state

## 35. Selecting the Next Ready Task

The next task must be:

- pending;
- dependency-complete;
- executable.

In [ ]:
def task_by_id(state: dict, task_id: str) -> dict:
    for task in state["tasks"]:
        if task["id"] == task_id:
            return task
    raise KeyError(task_id)


def task_dependencies_complete(state: dict, task: dict) -> bool:
    return all(
        task_by_id(state, dependency_id)["status"] == "completed"
        for dependency_id in task["depends_on"]
    )


def next_ready_task(state: dict):
    for task in state["tasks"]:
        if (
            task["status"] == "pending"
            and task_dependencies_complete(state, task)
        ):
            return task
    return None


next_ready_task(agent_state)

## 36. Defining Task Actions

We create deterministic task handlers.

In a real LLM-powered agent:

- a model may choose the tool;
- a runtime validates the choice;
- the tool performs the work.

In [ ]:
def collect_requirements(state: dict) -> dict:
    required = state["requirements"]

    if required["duration_minutes"] <= 0:
        return {
            "status": "error",
            "type": "input_error",
            "message": "Duration must be positive."
        }

    return {
        "status": "success",
        "observation": "Workshop requirements are available."
    }


def draft_agenda(state: dict) -> dict:
    sections = [
        "Introduction",
        "Concept explanation",
        "Demonstration",
        "Activity"
        # Recap intentionally omitted to trigger evaluation.
    ]

    state["agenda"] = {
        "duration_minutes": state["requirements"]["duration_minutes"],
        "sections": sections
    }

    return {
        "status": "success",
        "observation": "Draft agenda created."
    }


def validate_agenda(state: dict) -> dict:
    agenda = state["agenda"]
    required_sections = state["requirements"]["required_sections"]

    missing = [
        section
        for section in required_sections
        if section not in agenda["sections"]
    ]

    if missing:
        return {
            "status": "evaluation_failed",
            "severity": "minor",
            "missing_sections": missing
        }

    return {
        "status": "success",
        "observation": "Agenda passed validation."
    }

## 37. Revision Logic

The first draft intentionally omits “Recap”.

The evaluator should identify the issue and trigger a local revision.

In [ ]:
def revise_agenda(state: dict, evaluation: dict) -> dict:
    missing_sections = evaluation.get("missing_sections", [])

    for section in missing_sections:
        if section not in state["agenda"]["sections"]:
            state["agenda"]["sections"].append(section)

    state["counters"]["revisions"] += 1

    return {
        "status": "success",
        "observation": (
            "Agenda revised by adding: "
            + ", ".join(missing_sections)
        )
    }

## 38. Task Dispatcher

The dispatcher maps task IDs to handlers.

This is similar to a tool registry.

In [ ]:
task_handlers = {
    "T1": collect_requirements,
    "T2": draft_agenda,
    "T3": validate_agenda
}


def execute_task(state: dict, task: dict) -> dict:
    handler = task_handlers.get(task["id"])

    if handler is None:
        return {
            "status": "error",
            "type": "unknown_task",
            "message": f"No handler for {task['id']}"
        }

    return handler(state)

## 39. Recording Events

Each turn should preserve:

- task;
- result;
- action taken;
- state transition.

In [ ]:
def record_history(
    state: dict,
    task: dict,
    result: dict,
    action_taken: str
) -> None:
    state["history"].append({
        "turn": len(state["history"]) + 1,
        "task_id": task["id"],
        "task_name": task["name"],
        "result": result,
        "action_taken": action_taken
    })

## 40. Integrated Runtime

The runtime will:

1. select the next ready task;
2. execute it;
3. evaluate the result;
4. revise when needed;
5. enforce limits;
6. mark tasks complete;
7. stop when finished.

In [ ]:
def run_plan_act_evaluate(state: dict) -> dict:
    max_turns = state["limits"]["max_turns"]

    for turn in range(1, max_turns + 1):
        task = next_ready_task(state)

        if task is None:
            if all(t["status"] == "completed" for t in state["tasks"]):
                state["status"] = "completed"
            else:
                state["status"] = "blocked"
            break

        result = execute_task(state, task)

        if result["status"] == "success":
            task["status"] = "completed"
            record_history(
                state,
                task,
                result,
                action_taken="marked_completed"
            )

        elif result["status"] == "evaluation_failed":
            if (
                result.get("severity") == "minor"
                and state["counters"]["revisions"]
                < state["limits"]["max_revisions"]
            ):
                revision = revise_agenda(state, result)
                record_history(
                    state,
                    task,
                    result,
                    action_taken="revised_output"
                )

                # Re-run validation on the next loop.
                continue

            state["status"] = "evaluation_failed"
            record_history(
                state,
                task,
                result,
                action_taken="stopped"
            )
            break

        else:
            state["counters"]["tool_errors"] += 1
            record_history(
                state,
                task,
                result,
                action_taken="error_recorded"
            )

            if (
                state["counters"]["tool_errors"]
                >= state["limits"]["max_tool_errors"]
            ):
                state["status"] = "failed"
                break
    else:
        state["status"] = "turn_limit_reached"

    return state


final_state = run_plan_act_evaluate(agent_state)
final_state

## 41. Inspecting the Execution Trace

The trace should show:

1. requirements collected;
2. agenda drafted;
3. validation failed;
4. agenda revised;
5. validation passed;
6. goal completed.

In [ ]:
for event in final_state["history"]:
    print(f"Turn {event['turn']}: {event['task_id']} - {event['task_name']}")
    print("Result:", event["result"])
    print("Action:", event["action_taken"])
    print("-" * 70)

print("Final status:", final_state["status"])
print("Final agenda:", final_state["agenda"])

## 42. What the Integrated Example Demonstrates

The runtime contains several agentic patterns:

### Planning

Tasks and dependencies are explicit.

### Acting

Task handlers perform work.

### Evaluating

The agenda is validated against requirements.

### Reflection

A failure leads to a targeted revision.

### Minimal repair

Only the missing section is added.

### State

Progress, counters, agenda, and history are preserved.

### Stopping

The loop ends when all tasks complete or limits are exceeded.

## 43. Where an LLM Would Fit

In the deterministic example, Python functions made decisions.

An LLM-powered version could use a model to:

- decompose the goal;
- select the next task;
- draft the agenda;
- interpret evaluator feedback;
- propose a revised plan.

Deterministic software should still handle:

- dependency checks;
- counters;
- permission checks;
- schema validation;
- limit enforcement;
- and critical business rules.

```text
LLM flexibility
        +
Deterministic control
        =
More reliable agent
```

## 44. Common Failure Modes in Planning Agents

### Over-planning

The agent creates a large plan before enough information is available.

### Under-planning

The agent acts immediately and misses important dependencies.

### Plan rigidity

The agent follows an invalid plan after conditions change.

### Plan churn

The agent repeatedly rewrites a valid plan.

### Duplicate action

The agent repeats a completed tool call.

### Premature completion

The agent claims success before evaluation.

### Infinite revision

The agent keeps modifying output without limits.

### Wrong abstraction level

Tasks are too broad or too narrow.

## 45. Planning Heuristics

Useful heuristics include:

1. plan only as far as current information supports;
2. keep tasks independently verifiable;
3. record dependencies explicitly;
4. re-plan only when a trigger occurs;
5. preserve completed valid work;
6. prefer local repair before global re-planning;
7. evaluate important outputs;
8. limit retries and revisions;
9. stop when further progress requires user input;
10. never claim success without evidence.

## 46. Human-in-the-Loop Decision Points

Human review is useful when:

- the goal is ambiguous;
- several valid plans have meaningful trade-offs;
- a decision affects money or rights;
- irreversible action is proposed;
- safety risk is high;
- evaluation remains uncertain;
- limits are exhausted;
- policy requires approval.

The human should receive:

- current goal;
- completed work;
- proposed action;
- alternatives;
- risks;
- and the exact approval requested.

## Knowledge Check 6

1. What stages exist in a plan–act–evaluate loop?
2. What triggered revision in the Python example?
3. Why was the revision local rather than global?
4. Which controls remained deterministic?
5. Where could an LLM be inserted?
6. What is premature completion?
7. What is plan churn?
8. When should the system ask a human?
9. Why should approval requests include alternatives and risks?

## 47. Mini Lab — Build a Research Planning Agent

Design a planning agent for:

> “Create a short report comparing three AI agent frameworks.”

Your design should include:

### Goal

What exactly must be produced?

### Plan

Possible tasks:

1. define comparison criteria;
2. collect sources;
3. extract evidence;
4. compare frameworks;
5. draft report;
6. validate citations;
7. revise;
8. finish.

### Dependencies

Which tasks require previous tasks?

### Evaluators

How will you check:

- three frameworks are covered;
- citations exist;
- claims are supported;
- uncertainty is identified?

### Re-planning triggers

What if one framework has insufficient documentation?

### Stopping conditions

When is the report complete?

## 48. Architecture Worksheet

| Design Area | Your Answer |
|---|---|
| Goal | |
| Initial plan | |
| Atomic tasks | |
| Dependencies | |
| Ready-task rule | |
| Tools | |
| Evaluation criteria | |
| Retryable failures | |
| Non-retryable failures | |
| Re-planning triggers | |
| Revision limit | |
| Retry limit | |
| Human approval point | |
| Success evidence | |
| Graceful degradation output | |

## 49. Programming Exercises

### Exercise 1 — Dependency Graph

Add two parallel tasks to the integrated runtime:

- create participant worksheet;
- create feedback form.

Both should depend on requirement collection.

### Exercise 2 — Global Re-Planning

Add an observation:

```text
Workshop duration reduced from 120 to 60 minutes.
```

Revise all agenda-related tasks.

### Exercise 3 — Retry Policy

Create a tool that fails randomly and add bounded retries.

### Exercise 4 — Error Budget

Stop execution after:

- two tool errors;
- two revisions;
- one re-plan.

### Exercise 5 — Independent Evaluator

Create a separate evaluator function that scores:

- completeness;
- timing;
- clarity;
- safety.

### Exercise 6 — Graceful Degradation

Return a final report containing:

- completed tasks;
- failed tasks;
- blocked tasks;
- recommended next action.

## 50. Discussion Questions

1. Should an agent create a complete plan before taking any action?
2. When is planning overhead greater than its value?
3. Can deterministic workflows re-plan?
4. Should the same model create and evaluate an answer?
5. How many revisions should be allowed?
6. Should every failure trigger a re-plan?
7. What evidence is sufficient to mark a task complete?
8. How should an agent explain a changed plan?
9. When does retry become wasteful?
10. Which parts of planning should remain deterministic?

## 51. Common Misconceptions

### “Planning means writing a long list.”

A useful plan captures actionable tasks, dependencies, and success conditions.

### “Reflection means revealing hidden thoughts.”

Reflection can be implemented using explicit criteria and observable evaluation results.

### “Every failure should be retried.”

Only retry failures likely to be temporary.

### “The agent should always repair itself.”

Some failures require user input or human approval.

### “A model evaluator guarantees correctness.”

Model evaluators are probabilistic and need rubrics and controls.

### “Re-planning from scratch is safest.”

It may discard valid work and repeat costly actions.

### “More revisions always improve quality.”

Repeated revisions can reduce consistency and waste resources.

## 52. End-of-Part Quiz

1. Reasoning is best described as:  
   A. Storing files  
   B. Selecting useful next actions from available information  
   C. Running every tool  
   D. Writing a long answer

2. Planning primarily defines:  
   A. Future actions toward a goal  
   B. Model parameters  
   C. Database indexes  
   D. User permissions only

3. A task is ready when:  
   A. It is first in the list  
   B. Dependencies and requirements are satisfied  
   C. It has the longest description  
   D. It has already completed

4. Minimal repair means:  
   A. Delete the plan  
   B. Change only the affected valid portion  
   C. Restart every task  
   D. Ignore the failure

5. A good re-planning trigger is:  
   A. Random preference  
   B. Required resource unavailable  
   C. Every successful action  
   D. Every model token

6. Deterministic evaluation is preferred when:  
   A. Criteria can be checked exactly  
   B. Quality is entirely subjective  
   C. No schema exists  
   D. No output is produced

7. Reflection should result in:  
   A. A control-flow decision  
   B. More hidden text only  
   C. Unlimited revision  
   D. Removal of state

8. Which failure is most retryable?  
   A. Permission denied  
   B. Invalid password  
   C. Temporary network timeout  
   D. Prohibited action

9. Graceful degradation means:  
   A. Claim full success  
   B. Preserve partial value and report limitations  
   C. Hide failures  
   D. Continue forever

10. The strongest architecture combines:  
    A. LLM decisions with deterministic control and evaluation  
    B. Unlimited re-planning  
    C. No execution limits  
    D. Only one final model call

<details>
<summary><strong>Answer key</strong></summary>

1-B, 2-A, 3-B, 4-B, 5-B, 6-A, 7-A, 8-C, 9-B, 10-A

</details>

## 53. Summary

This notebook studied how modern agents make and improve decisions.

### Core ideas

```text
Goal
  ↓
Decompose
  ↓
Plan
  ↓
Select Ready Task
  ↓
Act
  ↓
Observe
  ↓
Evaluate
  ↓
Accept, Revise, Retry, Re-plan, Ask, or Stop
```

### Key lessons

- Reasoning selects useful next actions.
- Planning organises future work.
- Complex goals should be decomposed into verifiable tasks.
- Dependencies determine valid execution order.
- Dynamic plans change only when justified by observations.
- Minimal repair preserves valid completed work.
- Reflection should use explicit criteria.
- Deterministic evaluation is preferred when possible.
- Model-based evaluation requires rubrics and controls.
- Retries must be bounded and failure-aware.
- Error budgets prevent infinite recovery loops.
- Graceful degradation preserves partial value.
- Human intervention is appropriate when uncertainty or risk remains high.

## 54. Preview of Notebook 1.3 — Part 3

Part 3 will focus on the agent's capabilities and memory systems:

- tool design;
- function calling;
- tool schemas;
- tool selection;
- API tools;
- database tools;
- file tools;
- search tools;
- short-term state;
- working memory;
- long-term memory;
- episodic memory;
- semantic memory;
- memory retrieval;
- and memory safety.

The central question will be:

> How does an agent access the outside world and retain the information needed to complete long-running tasks?

---

**End of Notebook 1.3 — Part 2**